# Inspect Databricks traces

Saves every span from the first two GW1 runs into txt files in this folder:

- `trace_run1.txt` — first try (`cc41d1a308c44f6fbba4db915fa5c466`)
- `trace_run2.txt` — second try (`4c9ac301689946e4967e22534bce913a`)


In [9]:
from pathlib import Path
import json
import os
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

import mlflow

from fpl_sim.tracking.mlflow_tracker import load_yaml, resolve_experiment, resolve_tracking_uri

print("repo", ROOT)


repo /Users/ojonyeagwu/Desktop/fpl_benchmark


In [10]:
config = load_yaml("configs/baseline.yaml")
uri = resolve_tracking_uri()
mlflow.set_tracking_uri(uri)
if os.environ.get("MLFLOW_REGISTRY_URI"):
    mlflow.set_registry_uri(os.environ["MLFLOW_REGISTRY_URI"])

exp_id, exp_name = resolve_experiment(config, uri)
if exp_id:
    experiment = mlflow.set_experiment(experiment_id=exp_id)
else:
    experiment = mlflow.set_experiment(exp_name)
EXPERIMENT_ID = experiment.experiment_id
print("experiment", experiment.name, EXPERIMENT_ID)


experiment /Users/<user>/fpl_sim <experiment-id>


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


In [11]:
RUN_IDS = [
    "cc41d1a308c44f6fbba4db915fa5c466",  # first try
    "4c9ac301689946e4967e22534bce913a",  # second try
]
OUT_DIR = ROOT / "notebooks"


def load_traces(run_id: str):
    traces = mlflow.search_traces(
        locations=[EXPERIMENT_ID],
        run_id=run_id,
        include_spans=True,
        return_type="list",
        max_results=50,
        order_by=["timestamp_ms DESC"],
    )
    return list(traces or [])


def dump(value):
    return json.dumps(value, default=str, indent=2)


def format_run(run_index: int, run_id: str, traces) -> str:
    lines = [
        "=" * 80,
        f"RUN {run_index}: {run_id}",
        "=" * 80,
        f"{len(traces)} trace(s)",
        "",
    ]
    for trace in traces:
        spans = sorted(trace.data.spans, key=lambda s: s.start_time_ns or 0)
        lines.append(
            f"trace_id={trace.info.trace_id} state={trace.info.state} spans={len(spans)}"
        )
        for n, span in enumerate(spans, start=1):
            lines.append("")
            lines.append(f"--- span {n}: {span.name} ({span.span_type}) ---")
            lines.append("inputs:")
            lines.append(dump(span.inputs))
            lines.append("outputs:")
            lines.append(dump(span.outputs))
        lines.append("")
    return "\n".join(lines)


for i, run_id in enumerate(RUN_IDS, start=1):
    traces = load_traces(run_id)
    path = OUT_DIR / f"trace_run{i}.txt"
    path.write_text(format_run(i, run_id, traces))
    print(f"wrote {path} ({path.stat().st_size} bytes, {len(traces)} trace(s))")


wrote /Users/ojonyeagwu/Desktop/fpl_benchmark/notebooks/trace_run1.txt (47896 bytes, 1 trace(s))
wrote /Users/ojonyeagwu/Desktop/fpl_benchmark/notebooks/trace_run2.txt (60449 bytes, 1 trace(s))
